# OOA model with nanochat: pretrain, then train on our use case

Runs on a free Colab GPU (T4). Order: **Cell 1 config -> Cell 2 setup -> Cells 3-5 pretrain -> Cells 6-8 OOA training and test**.
Everything is saved to your Google Drive, so if Colab disconnects, reconnect and **run all cells again**: finished steps are
skipped and pretraining resumes from the last checkpoint.

Set Runtime -> Change runtime type -> GPU first. Start with `PROFILE = "smoke"` (about 15 minutes) to check the whole pipeline,
then switch to `"t4"` for the real run.

Honest expectations: a free T4 pretrains a small model (tens of millions of parameters, a few hundred million tokens).
It will learn English structure and JSON well enough to be fine-tuned, but it is not a strong general model.
This notebook was written without a GPU to test on, so if a cell errors, send me the error text.

In [ ]:
#@title 1. Config
PROFILE = "smoke"          # "smoke" = quick pipeline check, "t4" = real free-Colab run, "a100" = Colab Pro (A100/L4)
DRIVE_DIR = "/content/drive/MyDrive/ooa-nanochat"   # checkpoints, data and tokenizer live here
NANOCHAT_COMMIT = "92d63d4e8bb4df75c3b71618f31ddde2378b2bcd"   # version this notebook was written against
PLATFORM_REPO = "https://github.com/OneOfficeautomation-Private-Limited/OneOfficeAutomation-Platform"
VOCAB_SIZE = 16384         # tokenizer size (nanochat default is 32768; smaller suits a small model)

PROFILES = {
  #           depth, seq, device_batch, total_batch_tokens, target_tokens, data_shards, tok_chars
  "smoke": dict(depth=4,  seq=512,  dbs=16, total_batch=16384,  tokens=5_000_000,    shards=2, tok_chars=100_000_000),
  "t4":    dict(depth=8,  seq=1024, dbs=8,  total_batch=65536,  tokens=300_000_000,  shards=8, tok_chars=500_000_000),
  "a100":  dict(depth=12, seq=2048, dbs=16, total_batch=262144, tokens=1_000_000_000,shards=16,tok_chars=1_000_000_000),
}
P = PROFILES[PROFILE]
NUM_ITERS = P["tokens"] // P["total_batch"]

# SFT (training on our use case)
SFT_SEQ = 2048             # flows longer than this get cut; the T4 cannot afford much more
SFT_DEVICE_BATCH = 4
SFT_EPOCHS = 2 if PROFILE == "smoke" else 6      # passes over the OOA data (the seed set is small)
FLOW_REPEAT = 15           # each flow example is repeated this many times per pass so flows are about a third of the data
SFT_ITERS = 400            # upper limit; training also stops when the data runs out
SMOLTALK_ROWS = 3000       # a little general chat data mixed in so the model still talks; 0 = OOA data only
MAX_FLOW_CHARS = 9000      # skip training flows longer than this (about SFT_SEQ tokens)
print(PROFILE, P, "iterations:", NUM_ITERS)

In [ ]:
#@title 2. Setup: GPU, Drive, nanochat, dependencies
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU"
cap = torch.cuda.get_device_capability()
print(torch.cuda.get_device_name(0), "compute capability", cap)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ["NANOCHAT_BASE_DIR"] = DRIVE_DIR + "/nanochat_base"      # checkpoints, tokenizer, data, all persistent
# T4 has no bf16: use fp16 with loss scaling (supported by nanochat). If you see NaN losses, set "float32" (slower).
os.environ["NANOCHAT_DTYPE"] = "bfloat16" if cap >= (8, 0) else "float16"
os.environ["OMP_NUM_THREADS"] = "1"
print("compute dtype:", os.environ["NANOCHAT_DTYPE"])

if not os.path.exists("/content/nanochat"):
    !git clone -q https://github.com/karpathy/nanochat /content/nanochat
!cd /content/nanochat && git checkout -q {NANOCHAT_COMMIT}
# Keep Colab's own torch; install only nanochat's other dependencies
!pip -q install rustbpe tiktoken kernels filelock pyarrow psutil requests wandb jsonschema
%cd /content/nanochat

## Pretraining (general language, on nanochat's web-text dataset)

In [ ]:
#@title 3. Download pretraining data and train the tokenizer (skipped if already done)
tok_pkl = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "tokenizer", "tokenizer.pkl")
!python -m nanochat.dataset -n {P["shards"]}
if not os.path.exists(tok_pkl):
    !python -m scripts.tok_train --max-chars={P["tok_chars"]} --vocab-size={VOCAB_SIZE}
    !python -m scripts.tok_eval
else:
    print("tokenizer already trained:", tok_pkl)

In [ ]:
#@title 4. Pretrain (resumes automatically after a disconnect)
import glob, re
ckpt_dir = os.path.join(os.environ["NANOCHAT_BASE_DIR"], "base_checkpoints", f"d{P['depth']}")
steps = sorted(int(re.search(r"model_(\d+)\.pt", f).group(1)) for f in glob.glob(ckpt_dir + "/model_*.pt"))
resume = f"--resume-from-step={steps[-1]}" if steps else ""
if steps and steps[-1] >= NUM_ITERS:
    print("Pretraining already finished at step", steps[-1])
else:
    print("resuming from", steps[-1] if steps else "scratch")
    !python -m scripts.base_train --depth={P["depth"]} --head-dim=64 --window-pattern=L \
        --max-seq-len={P["seq"]} --device-batch-size={P["dbs"]} --total-batch-size={P["total_batch"]} \
        --num-iterations={NUM_ITERS} --eval-every=500 --eval-tokens={P["dbs"]*P["seq"]*20} \
        --core-metric-every=-1 --sample-every=1000 --save-every=250 --run=dummy {resume}

In [ ]:
#@title 5. Quick look at the pretrained model (it continues text; it does not follow instructions yet)
!python -m scripts.base_eval --eval=bpb,sample --device-batch-size=1 --split-tokens=65536 2>&1 | tail -25

## Training on our use case (OOA flows)

Builds training data from the platform repo with no LLM: requests to Flow JSON from the gold flows (3 wordings each),
plus catalog questions ("which action does X", "what inputs does Y take"). Some workflow families are **held out** and
never trained on; they are used in the final test. This seed set is small (about 60 flow examples): replace
`train.jsonl` with the full dialogue set when you have it.

In [ ]:
#@title 6. Build OOA training data
import json
os.makedirs("/content/ooa", exist_ok=True)
FILES = {"build_catalog.py": "\"\"\"Build the OOA action catalog from the platform's generated toolkit reference pages.\n\nUsage:\n    python build_catalog.py PLATFORM_REPO_DIR [OUT_JSONL]\n\nReads PLATFORM_REPO_DIR/docs/reference/toolkits/*.md (generated from `ooa manifest`) and writes one JSON object\nper action / condition / loop type:\n    {id, kind, toolkit, toolkit_id, name, description, inputs[], outputs[], notes}\nPure standard library; no platform install needed.\n\"\"\"\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nENTRY = re.compile(r\"^### (.+)$\")\nID_LINE = re.compile(r\"^`([^`]+)` \\((action|condition|loop type)\\)\\s*$\")\nHEAD = re.compile(r\"^# (.+)$\")\nTK_LINE = re.compile(r\"^Toolkit `([^`]+)`, version ([^.]+(?:\\.[^.]+)*)\\.\\s*(.*)$\")\n\n\ndef split_row(line):\n    # table cells; \"\\|\" is an escaped pipe inside a cell\n    cells = re.split(r\"(?<!\\\\)\\|\", line.strip().strip(\"|\"))\n    return [c.strip().replace(\"\\\\|\", \"|\") for c in cells]\n\n\ndef parse_table(lines, i):\n    \"\"\"lines[i] is the header row. Returns (rows as dicts, next index).\"\"\"\n    header = split_row(lines[i])\n    i += 2  # skip the --- separator\n    rows = []\n    while i < len(lines) and lines[i].startswith(\"|\"):\n        cells = split_row(lines[i])\n        rows.append(dict(zip(header, cells)))\n        i += 1\n    return rows, i\n\n\ndef parse_file(path):\n    lines = path.read_text(encoding=\"utf-8\").splitlines()\n    toolkit = \"\"\n    toolkit_id = \"\"\n    entries = []\n    cur = None\n    i = 0\n    while i < len(lines):\n        line = lines[i]\n        m = HEAD.match(line)\n        if m and not toolkit:\n            toolkit = m.group(1)\n        m = TK_LINE.match(line)\n        if m:\n            toolkit_id = m.group(1)\n        m = ENTRY.match(line)\n        if m:\n            cur = {\"name\": m.group(1), \"toolkit\": toolkit, \"toolkit_id\": toolkit_id,\n                   \"inputs\": [], \"outputs\": [], \"description\": \"\", \"notes\": \"\"}\n            entries.append(cur)\n            i += 1\n            continue\n        if cur is not None:\n            m = ID_LINE.match(line)\n            if m:\n                cur[\"id\"], cur[\"kind\"] = m.group(1), m.group(2)\n                # description = next non-empty line\n                j = i + 1\n                while j < len(lines) and not lines[j].strip():\n                    j += 1\n                if j < len(lines) and not lines[j].startswith((\"**\", \"|\", \"#\")):\n                    cur[\"description\"] = lines[j].strip()\n            elif line.strip() == \"**Settings**\":\n                j = i + 1\n                while j < len(lines) and not lines[j].startswith(\"|\"):\n                    j += 1\n                rows, i = parse_table(lines, j)\n                for r in rows:\n                    cur[\"inputs\"].append({\n                        \"name\": r.get(\"Name\", \"\").strip(\"`\"),\n                        \"label\": r.get(\"Field\", \"\"),\n                        \"type\": r.get(\"Type\", \"\"),\n                        \"required\": r.get(\"Required\", \"\") == \"Yes\",\n                        \"default\": r.get(\"Default\", \"\").strip(\"`\"),\n                        \"description\": r.get(\"Description\", \"\"),\n                    })\n                continue\n            elif line.strip() == \"**Results**\":\n                j = i + 1\n                while j < len(lines) and not lines[j].startswith(\"|\"):\n                    j += 1\n                rows, i = parse_table(lines, j)\n                for r in rows:\n                    cur[\"outputs\"].append({\n                        \"name\": r.get(\"Name\", \"\").strip(\"`\"),\n                        \"type\": r.get(\"Type\", \"\"),\n                        \"description\": r.get(\"Description\", \"\"),\n                    })\n                continue\n            elif line.startswith(\"*\") and line.endswith(\"*\") and len(line) > 2:\n                cur[\"notes\"] = line.strip(\"*\")\n        i += 1\n    return [e for e in entries if \"id\" in e]\n\n\ndef main():\n    repo = Path(sys.argv[1])\n    out = Path(sys.argv[2]) if len(sys.argv) > 2 else Path(__file__).with_name(\"catalog.jsonl\")\n    pages = sorted((repo / \"docs/reference/toolkits\").glob(\"*.md\"))\n    pages = [p for p in pages if p.name != \"README.md\"]\n    total = 0\n    with out.open(\"w\", encoding=\"utf-8\") as f:\n        for p in pages:\n            for e in parse_file(p):\n                f.write(json.dumps(e, ensure_ascii=False) + \"\\n\")\n                total += 1\n    print(f\"{total} entries from {len(pages)} toolkit pages -> {out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "build_sft_data.py": "\"\"\"Build seed SFT data for nanochat from the OOA platform repo. No LLM, no network beyond the clone.\n\nUsage: python build_sft_data.py PLATFORM_REPO OUT_DIR [MAX_CHARS]\nWrites OUT_DIR/{catalog.jsonl,train.jsonl,val.jsonl,eval.jsonl}\n  train/val : {\"messages\":[{\"role\":\"user\"|\"assistant\",\"content\":...}]}  (nanochat format)\n  eval      : {\"family\",\"prompt\",\"gold_flow\"}  held-out workflow families, never in train/val\nDialogue types: (1) request -> minified Flow JSON from the gold flows (3 prompt wordings each),\n                (2) catalog lookups: which action does X / what inputs does an action take.\nFlows longer than MAX_CHARS (default 12000) are skipped so they are not truncated at SFT length.\n\"\"\"\nimport glob, hashlib, json, os, random, re, sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\n\n\ndef family(path):\n    return re.sub(r\"-capture$\", \"\", os.path.basename(path)[:-len(\".flow.json\")])\n\n\ndef is_heldout(fam):\n    return int(hashlib.sha1(fam.encode()).hexdigest(), 16) % 4 == 0\n\n\nWORDINGS = [\n    \"{d}\",\n    \"Create an OOA flow for this: {d}\",\n    \"Build me an automation. {d}\",\n]\n\n\ndef lower_first(s):\n    return s[:1].lower() + s[1:] if s else s\n\n\ndef main():\n    repo, out = Path(sys.argv[1]), Path(sys.argv[2])\n    max_chars = int(sys.argv[3]) if len(sys.argv) > 3 else 12000\n    out.mkdir(parents=True, exist_ok=True)\n\n    # catalog\n    import importlib.util\n    here = Path(__file__).resolve().parent.parent / \"data/catalog/build_catalog.py\"\n    if not here.exists():\n        here = Path(__file__).resolve().parent / \"build_catalog.py\"  # notebook layout\n    spec = importlib.util.spec_from_file_location(\"bc\", here)\n    bc = importlib.util.module_from_spec(spec); spec.loader.exec_module(bc)\n    cat = []\n    for p in sorted((repo / \"docs/reference/toolkits\").glob(\"*.md\")):\n        if p.name != \"README.md\":\n            cat += bc.parse_file(p)\n    with open(out / \"catalog.jsonl\", \"w\", encoding=\"utf-8\") as f:\n        for e in cat:\n            f.write(json.dumps(e, ensure_ascii=False) + \"\\n\")\n\n    train, val, evals = [], [], []\n    rng = random.Random(7)\n    skipped = []\n    for path in sorted(glob.glob(str(repo / \"examples/**/*.flow.json\"), recursive=True)):\n        fam = family(path)\n        d = json.load(open(path, encoding=\"utf-8\"))\n        desc = (d.get(\"description\") or \"\").strip()\n        if not desc or fam == \"make-office-files\":\n            continue\n        flow = json.dumps(d, separators=(\",\", \":\"), ensure_ascii=False)\n        if is_heldout(fam):\n            if not path.endswith(\"-capture.flow.json\"):\n                evals.append({\"family\": fam, \"prompt\": desc, \"gold_flow\": os.path.relpath(path, repo)})\n            continue\n        if len(flow) > max_chars:\n            skipped.append((fam, len(flow)))\n            continue\n        for w in WORDINGS:\n            train.append({\"messages\": [{\"role\": \"user\", \"content\": w.format(d=desc)},\n                                       {\"role\": \"assistant\", \"content\": flow}]})\n\n    # catalog dialogues (actions only)\n    cat_rows = []\n    for e in cat:\n        if e[\"kind\"] != \"action\":\n            continue\n        req = [i[\"name\"] for i in e[\"inputs\"] if i[\"required\"]]\n        opt = [i[\"name\"] for i in e[\"inputs\"] if not i[\"required\"]]\n        desc = e[\"description\"].rstrip(\".\")\n        cat_rows.append({\"messages\": [\n            {\"role\": \"user\", \"content\": f\"Which OOA action would I use to {lower_first(desc)}?\"},\n            {\"role\": \"assistant\", \"content\": f\"Use `{e['id']}` ({e['toolkit']}: {e['name']}).\"}]})\n        cat_rows.append({\"messages\": [\n            {\"role\": \"user\", \"content\": f\"What inputs does {e['id']} take?\"},\n            {\"role\": \"assistant\", \"content\":\n                f\"`{e['id']}` requires: {', '.join(req) if req else 'nothing'}. \"\n                f\"Optional: {', '.join(opt[:12]) if opt else 'none'}.\"}]})\n    rng.shuffle(cat_rows)\n    cut = max(1, len(cat_rows) // 50)\n    val = cat_rows[:cut] + [r for r in train[::17]]\n    train = train + cat_rows[cut:]\n    rng.shuffle(train)\n\n    # separate files so the trainer can weight flows against the (much more numerous) catalog rows\n    flow_rows = [r for r in train if len(r[\"messages\"][1][\"content\"]) > 800 and r[\"messages\"][1][\"content\"].startswith(\"{\")]\n    cat_only = [r for r in train if r not in flow_rows]\n    for name, rows in ((\"train\", train), (\"train_flows\", flow_rows), (\"train_catalog\", cat_only),\n                       (\"val\", val), (\"eval\", evals)):\n        with open(out / f\"{name}.jsonl\", \"w\", encoding=\"utf-8\") as f:\n            for r in rows:\n                f.write(json.dumps(r, ensure_ascii=False) + \"\\n\")\n    print(f\"catalog {len(cat)} | train {len(train)} val {len(val)} eval {len(evals)} \"\n          f\"| flows skipped for length: {len(skipped)} {skipped[:6]}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "ooa_task.py": "\"\"\"nanochat Task that serves OOA dialogues from a JSONL file.\n\nCopy into nanochat/tasks/ (or add to PYTHONPATH) and use in scripts/chat_sft.py's train_tasks list, e.g.\n    from tasks.ooa import OOADialogues\n    train_tasks = [OOADialogues(\"data/dialogues/train.jsonl\"), ...]\nEach JSONL line: {\"messages\": [{\"role\": \"user\"|\"assistant\", \"content\": str}, ...]}\n - optional first message with role \"system\" (nanochat merges it into the first user turn)\n - roles must alternate user, assistant, ...; assistant content is the minified Flow JSON or a short reply\nFormat checked against nanochat tasks/smoltalk.py and Tokenizer.render_conversation (commit 92d63d4).\n\"\"\"\nimport json\nfrom tasks.common import Task\n\n\nclass OOADialogues(Task):\n    def __init__(self, path, **kwargs):\n        super().__init__(**kwargs)\n        with open(path, encoding=\"utf-8\") as f:\n            self.rows = [json.loads(l) for l in f if l.strip()]\n\n    @property\n    def eval_type(self):\n        return \"generative\"\n\n    def num_examples(self):\n        return len(self.rows)\n\n    def get_example(self, index):\n        messages = self.rows[index][\"messages\"]\n        rest = messages[1:] if messages[0][\"role\"] == \"system\" else messages\n        assert len(rest) >= 2\n        for i, m in enumerate(rest):\n            assert m[\"role\"] == (\"user\" if i % 2 == 0 else \"assistant\")\n            assert isinstance(m[\"content\"], str)\n        return {\"messages\": messages}\n", "flow_check.py": "\"\"\"Flow scorer: JSON-schema validity + every action/input exists in the catalog.\nUsage: python flow_check.py PLATFORM_REPO catalog.jsonl\nWhy not only `ooa validate`: the Python reference lacks most toolkits (browser, desktop...), so it rejects valid flows.\n\"\"\"\nimport json,glob,sys\nimport jsonschema\nP=sys.argv[1]; C=sys.argv[2]\nschema=json.load(open(P+'/contracts/flow-schema/flow.v1.json'))\ncat={}\nfor l in open(C):\n    e=json.loads(l); cat[e['id']]=e\ndef steps(n):\n    for s in n:\n        yield s\n        for k in ('children','else'):\n            v=s.get(k)\n            if isinstance(v,list): yield from steps(v)\n        for ei in s.get('elseIf',[]) or []:\n            yield from steps(ei.get('children',[]))\n        for c in s.get('catch',[]) or []:\n            yield from steps(c.get('children',[]))\n        if isinstance(s.get('finally'),list): yield from steps(s['finally'])\ntot=sok=0; unk=bad_in=0; unk_ids=set(); ex=[]\nfor f in sorted(glob.glob(P+'/examples/**/*.flow.json',recursive=True)):\n    d=json.load(open(f)); tot+=1\n    try: jsonschema.validate(d,schema); sok+=1\n    except Exception as e: ex.append((f.split('/')[-1],str(e)[:100]))\n    for s in steps(d.get('steps',[])):\n        a=s.get('action') or s.get('use') or s.get('type')\n        if not a or not isinstance(a,str) or '/' not in a: continue\n        if a not in cat: unk+=1; unk_ids.add(a); continue\n        names={i['name'] for i in cat[a]['inputs']}\n        for k in (s.get('args') or s.get('inputs') or {}):\n            if k not in names: bad_in+=1\nprint('flows',tot,'schema_ok',sok,'unknown actions',unk,sorted(unk_ids)[:8],'unknown inputs',bad_in); print(ex[:3])\n"}
for name, text in FILES.items():
    open(f"/content/ooa/{name}", "w", encoding="utf-8").write(text)
if not os.path.exists("/content/platform"):
    !git clone -q --depth 1 {PLATFORM_REPO} /content/platform
!python /content/ooa/build_sft_data.py /content/platform {DRIVE_DIR}/sft {MAX_FLOW_CHARS}
!cp /content/ooa/ooa_task.py /content/nanochat/tasks/ooa.py
!head -c 700 {DRIVE_DIR}/sft/train.jsonl

In [ ]:
#@title 7. Patch nanochat SFT to use our data, then train
import re
p = "/content/nanochat/scripts/chat_sft.py"
s = open(p).read()
if "OOADialogues" not in s:
    s = s.replace("from tasks.smoltalk import SmolTalk", "from tasks.smoltalk import SmolTalk\nfrom tasks.ooa import OOADialogues")
    a = s.index("train_tasks = [")
    b = s.index("\n", s.index("# total: 24K"))
    new = f"""train_tasks = ([OOADialogues("{DRIVE_DIR}/sft/train_flows.jsonl") for _ in range({FLOW_REPEAT})]
               + [OOADialogues("{DRIVE_DIR}/sft/train_catalog.jsonl")]) * {SFT_EPOCHS}
if {SMOLTALK_ROWS} > 0:
    train_tasks.append(SmolTalk(split="train", stop={SMOLTALK_ROWS}))
train_dataset = TaskMixture(train_tasks)
print0(f"OOA training mixture: {{len(train_dataset):,}} rows")
val_dataset = TaskMixture([OOADialogues("{DRIVE_DIR}/sft/val.jsonl")])"""
    s = s[:a] + new + s[b:]
    open(p, "w").write(s)
    print("patched chat_sft.py")
!python -m scripts.chat_sft --max-seq-len={SFT_SEQ} --device-batch-size={SFT_DEVICE_BATCH} \
    --total-batch-size={SFT_SEQ*SFT_DEVICE_BATCH*4} --num-iterations={SFT_ITERS} \
    --eval-every=100 --eval-tokens={SFT_SEQ*SFT_DEVICE_BATCH*8} --chatcore-every=-1 --run=dummy

In [ ]:
#@title 8. Test on held-out workflow families (never trained on)
import subprocess, json, sys
# chat_cli stops after 256 new tokens; flows are longer, so raise the limit
cli = "/content/nanochat/scripts/chat_cli.py"
t = open(cli).read()
t = t.replace('"max_tokens": 256', '"max_tokens": 3000')
open(cli, "w").write(t)
sys.path.insert(0, "/content/ooa")
import jsonschema
schema = json.load(open("/content/platform/contracts/flow-schema/flow.v1.json"))
catalog = {json.loads(l)["id"]: json.loads(l) for l in open(f"{DRIVE_DIR}/sft/catalog.jsonl")}

def steps(nodes):
    for s in nodes:
        yield s
        for k in ("children", "else"):
            if isinstance(s.get(k), list): yield from steps(s[k])
        for ei in s.get("elseIf") or []: yield from steps(ei.get("children", []))
        for c in s.get("catch") or []: yield from steps(c.get("children", []))
        if isinstance(s.get("finally"), list): yield from steps(s["finally"])

results = []
for row in map(json.loads, open(f"{DRIVE_DIR}/sft/eval.jsonl")):
    out = subprocess.run(["python", "-m", "scripts.chat_cli", "-i", "sft", "-p", row["prompt"], "-t", "0.2"],
                         capture_output=True, text=True, cwd="/content/nanochat").stdout
    text = out.split("Assistant:", 1)[-1].strip()
    r = {"family": row["family"], "json": False, "schema": False, "actions_known": False}
    try:
        flow = json.loads(text); r["json"] = True
        jsonschema.validate(flow, schema); r["schema"] = True
        acts = [s.get("action") for s in steps(flow.get("steps", [])) if isinstance(s.get("action"), str) and "/" in s["action"]]
        r["actions_known"] = all(a in catalog for a in acts)
    except Exception as e:
        r["error"] = str(e)[:80]
    results.append(r); print(r)
n = len(results)
print(f"\nheld-out flows: {n} | valid JSON {sum(r['json'] for r in results)} | schema OK {sum(r['schema'] for r in results)} | all actions exist {sum(r['actions_known'] for r in results)}")
print("\nTry it yourself:")
!cd /content/nanochat && python -m scripts.chat_cli -i sft -p "Which OOA action would I use to read a CSV file into a table?" -t 0.2

## After this run
- Checkpoints: `DRIVE_DIR/nanochat_base/base_checkpoints` (pretrained) and `chatsft_checkpoints` (OOA-trained).
- Weak results on the held-out test are expected with the seed data. The next lever is more and better dialogues
  (edits, error fixing, missing-input questions), validated with the scorer in `eval/flow_check.py`.
- For a stronger base, use `PROFILE = "t4"` and let it run (resume after disconnects), or an A100 on Colab Pro.